In [3]:
# ============================================================
#      STEP 8 — BUSINESS INSIGHTS & RECOMMENDATIONS
# ============================================================
# Goal:
# Convert the analysis results into actionable business
# insights and exactly 5 business recommendations.
#
# Outputs:
#   1. Business KPI summary
#   2. Customer insights
#   3. Product insights
#   4. Purchase behavior insights
#   5. Churn insights
#   6. RFM segment insights
#   7. Exactly 5 actionable recommendations
#   8. Business recommendations CSV
#   9. Executive summary TXT
#
# ============================================================


# ------------------------------------------------------------
# STEP 1 — IMPORT LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# STEP 2 — CREATE OUTPUT FOLDER
# ------------------------------------------------------------

output_folder = Path("step8_business_insights")

output_folder.mkdir(exist_ok=True)


# ============================================================
# STEP 3 — LOAD PHASE 7 DATA
# ============================================================

base_folder = Path("step7_powerbi_data")

fact_sales = pd.read_csv(
    base_folder / "fact_sales.csv"
)

customer_kpis = pd.read_csv(
    base_folder / "customer_kpis.csv"
)

monthly_sales = pd.read_csv(
    base_folder / "monthly_sales.csv"
)

category_performance = pd.read_csv(
    base_folder / "category_performance.csv"
)

payment_performance = pd.read_csv(
    base_folder / "payment_performance.csv"
)

day_performance = pd.read_csv(
    base_folder / "day_performance.csv"
)

hour_performance = pd.read_csv(
    base_folder / "hour_performance.csv"
)

time_period_performance = pd.read_csv(
    base_folder / "time_period_performance.csv"
)

rfm_segment_summary = pd.read_csv(
    base_folder / "rfm_segment_summary.csv"
)

segment_churn = pd.read_csv(
    base_folder / "segment_churn.csv"
)


# ------------------------------------------------------------
# STEP 4 — LOAD RFM DATA
# ------------------------------------------------------------

rfm = pd.read_csv(
    "customer_rfm_segments.csv"
)


# ============================================================
# STEP 5 — CONVERT NUMERIC COLUMNS
# ============================================================

numeric_columns = [
    "total_purchase_amount",
    "quantity",
    "product_price",
    "Revenue"
]

for col in numeric_columns:

    if col in fact_sales.columns:

        fact_sales[col] = pd.to_numeric(
            fact_sales[col],
            errors="coerce"
        )


# ============================================================
# SECTION A — OVERALL BUSINESS PERFORMANCE
# ============================================================

# ------------------------------------------------------------
# STEP 6 — CALCULATE MAIN KPIs
# ------------------------------------------------------------

total_revenue = fact_sales[
    "total_purchase_amount"
].sum()

total_transactions = len(fact_sales)

total_customers = fact_sales[
    "customer_id"
].nunique()

average_order_value = (
    total_revenue /
    total_transactions
)

total_quantity = fact_sales[
    "quantity"
].sum()

average_quantity = (
    total_quantity /
    total_transactions
)


# ------------------------------------------------------------
# STEP 7 — RETURN KPI
# ------------------------------------------------------------

if "Return_Flag" in fact_sales.columns:

    total_returns = fact_sales[
        "Return_Flag"
    ].sum()

    return_rate = (
        total_returns /
        total_transactions
    ) * 100

else:

    total_returns = 0
    return_rate = 0


# ------------------------------------------------------------
# STEP 8 — CHURN KPI
# ------------------------------------------------------------

if "churn" in customer_kpis.columns:

    churned_customers = (
        customer_kpis["churn"]
        .fillna(0)
        .sum()
    )

    churn_rate = (
        churned_customers /
        total_customers
    ) * 100

else:

    churned_customers = 0
    churn_rate = 0


# ------------------------------------------------------------
# STEP 9 — DISPLAY OVERALL KPIs
# ------------------------------------------------------------

business_kpis = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Total Transactions",
        "Total Revenue",
        "Average Order Value",
        "Total Quantity",
        "Average Quantity per Transaction",
        "Total Returns",
        "Return Rate (%)",
        "Churned Customers",
        "Churn Rate (%)"
    ],
    "Value": [
        total_customers,
        total_transactions,
        total_revenue,
        average_order_value,
        total_quantity,
        average_quantity,
        total_returns,
        return_rate,
        churned_customers,
        churn_rate
    ]
})

print("\n================ BUSINESS KPIs ================\n")

display(business_kpis)


# ============================================================
# SECTION B — PRODUCT INSIGHTS
# ============================================================

# ------------------------------------------------------------
# STEP 10 — BEST CATEGORY BY REVENUE
# ------------------------------------------------------------

best_category_row = category_performance.loc[
    category_performance["Revenue"].idxmax()
]

best_category = (
    best_category_row["product_category"]
)

best_category_revenue = (
    best_category_row["Revenue"]
)


# ------------------------------------------------------------
# STEP 11 — MOST TRANSACTED CATEGORY
# ------------------------------------------------------------

most_purchased_category_row = (
    category_performance.loc[
        category_performance["Transactions"].idxmax()
    ]
)

most_purchased_category = (
    most_purchased_category_row[
        "product_category"
    ]
)


# ------------------------------------------------------------
# STEP 12 — CATEGORY WITH HIGHEST RETURN RATE
# ------------------------------------------------------------

if "Return_Rate" in category_performance.columns:

    highest_return_category_row = (
        category_performance.loc[
            category_performance["Return_Rate"].idxmax()
        ]
    )

    highest_return_category = (
        highest_return_category_row[
            "product_category"
        ]
    )

    highest_return_rate = (
        highest_return_category_row[
            "Return_Rate"
        ]
    )

else:

    highest_return_category = "N/A"
    highest_return_rate = 0


# ------------------------------------------------------------
# STEP 13 — CREATE PRODUCT INSIGHT TABLE
# ------------------------------------------------------------

product_insights = pd.DataFrame({
    "Insight": [
        "Highest Revenue Category",
        "Most Purchased Category",
        "Highest Return Rate Category"
    ],
    "Value": [
        best_category,
        most_purchased_category,
        highest_return_category
    ],
    "Metric": [
        round(best_category_revenue, 2),
        int(
            most_purchased_category_row[
                "Transactions"
            ]
        ),
        round(highest_return_rate, 2)
    ]
})

print("\n================ PRODUCT INSIGHTS ================\n")

display(product_insights)


# ============================================================
# SECTION C — PURCHASE BEHAVIOR INSIGHTS
# ============================================================

# ------------------------------------------------------------
# STEP 14 — BEST DAY BY TRANSACTIONS
# ------------------------------------------------------------

best_day_row = day_performance.loc[
    day_performance["Transactions"].idxmax()
]

best_purchase_day = (
    best_day_row["Day_Name"]
)

best_day_transactions = (
    best_day_row["Transactions"]
)


# ------------------------------------------------------------
# STEP 15 — BEST HOUR BY TRANSACTIONS
# ------------------------------------------------------------

best_hour_row = hour_performance.loc[
    hour_performance["Transactions"].idxmax()
]

best_purchase_hour = (
    best_hour_row["Purchase_Hour"]
)

best_hour_transactions = (
    best_hour_row["Transactions"]
)


# ------------------------------------------------------------
# STEP 16 — BEST TIME PERIOD
# ------------------------------------------------------------

best_time_row = time_period_performance.loc[
    time_period_performance["Transactions"].idxmax()
]

best_time_period = (
    best_time_row["Time_Period"]
)

best_time_transactions = (
    best_time_row["Transactions"]
)


# ------------------------------------------------------------
# STEP 17 — BEST MONTH BY REVENUE
# ------------------------------------------------------------

best_month_row = monthly_sales.loc[
    monthly_sales["Revenue"].idxmax()
]

best_month = (
    best_month_row["Month_Year"]
)

best_month_revenue = (
    best_month_row["Revenue"]
)


# ------------------------------------------------------------
# STEP 18 — CREATE PURCHASE INSIGHT TABLE
# ------------------------------------------------------------

purchase_insights = pd.DataFrame({
    "Insight": [
        "Best Purchase Day",
        "Peak Purchase Hour",
        "Best Time Period",
        "Highest Revenue Month"
    ],
    "Value": [
        best_purchase_day,
        str(int(best_purchase_hour)) + ":00",
        best_time_period,
        best_month
    ],
    "Metric": [
        best_day_transactions,
        best_hour_transactions,
        best_time_transactions,
        best_month_revenue
    ]
})

print("\n================ PURCHASE BEHAVIOR INSIGHTS ================\n")

display(purchase_insights)


# ============================================================
# SECTION D — CUSTOMER VALUE INSIGHTS
# ============================================================

# ------------------------------------------------------------
# STEP 19 — TOP CUSTOMER
# ------------------------------------------------------------

if "Total_Spend" in customer_kpis.columns:

    top_customer_row = customer_kpis.loc[
        customer_kpis["Total_Spend"].idxmax()
    ]

    top_customer_id = (
        top_customer_row["customer_id"]
    )

    top_customer_spend = (
        top_customer_row["Total_Spend"]
    )

else:

    top_customer_id = "N/A"
    top_customer_spend = 0


# ------------------------------------------------------------
# STEP 20 — AVERAGE CUSTOMER SPEND
# ------------------------------------------------------------

if "Total_Spend" in customer_kpis.columns:

    average_customer_spend = (
        customer_kpis["Total_Spend"]
        .mean()
    )

else:

    average_customer_spend = 0


# ------------------------------------------------------------
# STEP 21 — AVERAGE CUSTOMER FREQUENCY
# ------------------------------------------------------------

if "Purchase_Frequency" in customer_kpis.columns:

    average_customer_frequency = (
        customer_kpis["Purchase_Frequency"]
        .mean()
    )

else:

    average_customer_frequency = 0


# ------------------------------------------------------------
# STEP 22 — CREATE CUSTOMER INSIGHT TABLE
# ------------------------------------------------------------

customer_insights = pd.DataFrame({
    "Insight": [
        "Top Customer",
        "Top Customer Spend",
        "Average Customer Spend",
        "Average Purchase Frequency"
    ],
    "Value": [
        top_customer_id,
        top_customer_spend,
        average_customer_spend,
        average_customer_frequency
    ]
})

print("\n================ CUSTOMER VALUE INSIGHTS ================\n")

display(customer_insights)


# ============================================================
# SECTION E — RFM SEGMENT INSIGHTS
# ============================================================

# ------------------------------------------------------------
# STEP 23 — LARGEST CUSTOMER SEGMENT
# ------------------------------------------------------------

largest_segment_row = rfm_segment_summary.loc[
    rfm_segment_summary["Customers"].idxmax()
]

largest_segment = (
    largest_segment_row["Customer_Segment"]
)

largest_segment_customers = (
    largest_segment_row["Customers"]
)


# ------------------------------------------------------------
# STEP 24 — HIGHEST REVENUE SEGMENT
# ------------------------------------------------------------

highest_revenue_segment_row = (
    rfm_segment_summary.loc[
        rfm_segment_summary["Revenue"].idxmax()
    ]
)

highest_revenue_segment = (
    highest_revenue_segment_row[
        "Customer_Segment"
    ]
)

highest_revenue_segment_value = (
    highest_revenue_segment_row[
        "Revenue"
    ]
)


# ------------------------------------------------------------
# STEP 25 — HIGHEST CHURN SEGMENT
# ------------------------------------------------------------

if "Churn_Rate" in segment_churn.columns:

    highest_churn_segment_row = (
        segment_churn.loc[
            segment_churn["Churn_Rate"].idxmax()
        ]
    )

    highest_churn_segment = (
        highest_churn_segment_row[
            "Customer_Segment"
        ]
    )

    highest_churn_rate = (
        highest_churn_segment_row[
            "Churn_Rate"
        ]
    )

else:

    highest_churn_segment = "N/A"
    highest_churn_rate = 0


# ------------------------------------------------------------
# STEP 26 — CREATE RFM INSIGHT TABLE
# ------------------------------------------------------------

rfm_insights = pd.DataFrame({
    "Insight": [
        "Largest Customer Segment",
        "Highest Revenue Segment",
        "Highest Churn Segment"
    ],
    "Value": [
        largest_segment,
        highest_revenue_segment,
        highest_churn_segment
    ],
    "Metric": [
        largest_segment_customers,
        highest_revenue_segment_value,
        highest_churn_rate
    ]
})

print("\n================ RFM INSIGHTS ================\n")

display(rfm_insights)


# ============================================================
# SECTION F — CHURN INSIGHTS
# ============================================================

# ------------------------------------------------------------
# STEP 27 — CHURNED REVENUE
# ------------------------------------------------------------

if {
    "churn",
    "Total_Spend"
}.issubset(customer_kpis.columns):

    churned_revenue = (
        customer_kpis.loc[
            customer_kpis["churn"] == 1,
            "Total_Spend"
        ].sum()
    )

    active_revenue = (
        customer_kpis.loc[
            customer_kpis["churn"] == 0,
            "Total_Spend"
        ].sum()
    )

else:

    churned_revenue = 0
    active_revenue = 0


# ------------------------------------------------------------
# STEP 28 — CHURNED REVENUE PERCENTAGE
# ------------------------------------------------------------

total_customer_revenue = (
    churned_revenue +
    active_revenue
)

if total_customer_revenue > 0:

    churned_revenue_percentage = (
        churned_revenue /
        total_customer_revenue
    ) * 100

else:

    churned_revenue_percentage = 0


# ------------------------------------------------------------
# STEP 29 — HIGH-VALUE AT-RISK CUSTOMERS
# ------------------------------------------------------------

if {
    "R_Score",
    "M_Score",
    "churn"
}.issubset(rfm.columns):

    high_value_at_risk = rfm[
        (
            (rfm["R_Score"] <= 2) &
            (rfm["M_Score"] >= 4) &
            (rfm["churn"] == 0)
        )
    ].copy()

else:

    high_value_at_risk = pd.DataFrame()


high_value_at_risk_count = len(
    high_value_at_risk
)


# ------------------------------------------------------------
# STEP 30 — CHURN INSIGHT TABLE
# ------------------------------------------------------------

churn_insights = pd.DataFrame({
    "Insight": [
        "Total Churned Customers",
        "Overall Churn Rate (%)",
        "Revenue from Churned Customers",
        "Churned Revenue Share (%)",
        "High-Value At-Risk Customers"
    ],
    "Value": [
        churned_customers,
        churn_rate,
        churned_revenue,
        churned_revenue_percentage,
        high_value_at_risk_count
    ]
})

print("\n================ CHURN INSIGHTS ================\n")

display(churn_insights)


# ============================================================
# SECTION G — PAYMENT INSIGHTS
# ============================================================

# ------------------------------------------------------------
# STEP 31 — MOST USED PAYMENT METHOD
# ------------------------------------------------------------

most_used_payment_row = (
    payment_performance.loc[
        payment_performance["Transactions"].idxmax()
    ]
)

most_used_payment = (
    most_used_payment_row[
        "payment_method"
    ]
)

most_used_payment_transactions = (
    most_used_payment_row[
        "Transactions"
    ]
)


# ------------------------------------------------------------
# STEP 32 — HIGHEST REVENUE PAYMENT METHOD
# ------------------------------------------------------------

highest_revenue_payment_row = (
    payment_performance.loc[
        payment_performance["Revenue"].idxmax()
    ]
)

highest_revenue_payment = (
    highest_revenue_payment_row[
        "payment_method"
    ]
)

highest_revenue_payment_value = (
    highest_revenue_payment_row[
        "Revenue"
    ]
)


# ------------------------------------------------------------
# STEP 33 — PAYMENT INSIGHT TABLE
# ------------------------------------------------------------

payment_insights = pd.DataFrame({
    "Insight": [
        "Most Used Payment Method",
        "Highest Revenue Payment Method"
    ],
    "Value": [
        most_used_payment,
        highest_revenue_payment
    ],
    "Metric": [
        most_used_payment_transactions,
        highest_revenue_payment_value
    ]
})

print("\n================ PAYMENT INSIGHTS ================\n")

display(payment_insights)


# ============================================================
# SECTION H — GENERATE EXACTLY 5 RECOMMENDATIONS
# ============================================================

# ------------------------------------------------------------
# STEP 34 — RECOMMENDATION 1
# Customer retention
# ------------------------------------------------------------

recommendation_1 = (
    f"Launch a targeted retention campaign for "
    f"high-value at-risk customers. "
    f"The analysis identified {high_value_at_risk_count:,} "
    f"customers who have high monetary value but low recent "
    f"engagement. Use personalized offers, reminders and "
    f"exclusive benefits to encourage repeat purchases."
)


# ------------------------------------------------------------
# STEP 35 — RECOMMENDATION 2
# RFM segmentation
# ------------------------------------------------------------

recommendation_2 = (
    f"Use RFM-based customer segmentation for personalized "
    f"marketing. Prioritize the '{highest_revenue_segment}' "
    f"segment for loyalty programs and cross-selling, while "
    f"using reactivation campaigns for low-recency segments."
)


# ------------------------------------------------------------
# STEP 36 — RECOMMENDATION 3
# Product strategy
# ------------------------------------------------------------

recommendation_3 = (
    f"Focus inventory and marketing efforts on the "
    f"'{best_category}' category because it generates the "
    f"highest revenue in the analyzed dataset. "
    f"Combine this category with cross-selling and bundle "
    f"offers to increase customer order value."
)


# ------------------------------------------------------------
# STEP 37 — RECOMMENDATION 4
# Purchase timing
# ------------------------------------------------------------

recommendation_4 = (
    f"Schedule promotional campaigns around the strongest "
    f"customer purchase period. The peak transaction period "
    f"is {best_purchase_day} at approximately "
    f"{int(best_purchase_hour)}:00. "
    f"Use this period for flash sales, personalized offers "
    f"and campaign reminders."
)


# ------------------------------------------------------------
# STEP 38 — RECOMMENDATION 5
# Churn reduction
# ------------------------------------------------------------

recommendation_5 = (
    f"Implement an early-warning churn system. "
    f"The overall churn rate is approximately "
    f"{churn_rate:.2f}%. "
    f"Customers showing declining recency and frequency "
    f"should receive automated retention messages before "
    f"they become fully inactive."
)


# ============================================================
# STEP 39 — CREATE RECOMMENDATIONS DATAFRAME
# ============================================================

recommendations = pd.DataFrame({
    "Recommendation_ID": [
        1,
        2,
        3,
        4,
        5
    ],
    "Priority": [
        "High",
        "High",
        "Medium",
        "Medium",
        "High"
    ],
    "Recommendation": [
        recommendation_1,
        recommendation_2,
        recommendation_3,
        recommendation_4,
        recommendation_5
    ]
})


# ------------------------------------------------------------
# STEP 40 — DISPLAY RECOMMENDATIONS
# ------------------------------------------------------------

print("\n====================================================")
print("5 ACTIONABLE BUSINESS RECOMMENDATIONS")
print("====================================================\n")

for _, row in recommendations.iterrows():

    print(
        f"{row['Recommendation_ID']}. "
        f"[{row['Priority']}] "
        f"{row['Recommendation']}\n"
    )


# ============================================================
# SECTION I — CREATE EXECUTIVE SUMMARY
# ============================================================

# ------------------------------------------------------------
# STEP 41 — EXECUTIVE SUMMARY TEXT
# ------------------------------------------------------------

executive_summary = f"""
============================================================
CUSTOMER BEHAVIOR ANALYSIS — EXECUTIVE SUMMARY
============================================================

BUSINESS PERFORMANCE
--------------------
Total Customers       : {total_customers:,}
Total Transactions    : {total_transactions:,}
Total Revenue         : {total_revenue:,.2f}
Average Order Value   : {average_order_value:,.2f}
Total Quantity Sold   : {total_quantity:,}
Return Rate           : {return_rate:.2f}%
Churn Rate            : {churn_rate:.2f}%

CUSTOMER INSIGHTS
-----------------
Largest Customer Segment:
{largest_segment}

Highest Revenue Segment:
{highest_revenue_segment}

Top Customer:
{top_customer_id}

Average Customer Spend:
{average_customer_spend:,.2f}

PRODUCT INSIGHTS
----------------
Highest Revenue Category:
{best_category}

Most Purchased Category:
{most_purchased_category}

Highest Return Rate Category:
{highest_return_category}

PURCHASE BEHAVIOR
-----------------
Best Purchase Day:
{best_purchase_day}

Peak Purchase Hour:
{int(best_purchase_hour)}:00

Best Time Period:
{best_time_period}

Highest Revenue Month:
{best_month}

PAYMENT BEHAVIOR
----------------
Most Used Payment Method:
{most_used_payment}

Highest Revenue Payment Method:
{highest_revenue_payment}

CHURN INSIGHTS
--------------
Churned Customers:
{churned_customers:,}

Churn Rate:
{churn_rate:.2f}%

Revenue from Churned Customers:
{churned_revenue:,.2f}

High-Value At-Risk Customers:
{high_value_at_risk_count:,}

TOP 5 BUSINESS RECOMMENDATIONS
------------------------------

1. Target high-value at-risk customers with personalized
   retention campaigns.

2. Use RFM segmentation for personalized marketing,
   loyalty programs and customer reactivation.

3. Prioritize the highest-revenue product category for
   inventory, cross-selling and promotional campaigns.

4. Schedule promotions around peak customer purchase
   periods.

5. Implement an early-warning churn system based on
   recency and purchase frequency.

============================================================
END OF EXECUTIVE SUMMARY
============================================================
"""


# ------------------------------------------------------------
# STEP 42 — SAVE EXECUTIVE SUMMARY
# ------------------------------------------------------------

with open(
    output_folder / "executive_summary.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(executive_summary)


# ============================================================
# SECTION J — SAVE ALL INSIGHTS
# ============================================================

# ------------------------------------------------------------
# STEP 43 — SAVE BUSINESS KPIs
# ------------------------------------------------------------

business_kpis.to_csv(
    output_folder / "business_kpis.csv",
    index=False
)


# ------------------------------------------------------------
# STEP 44 — SAVE PRODUCT INSIGHTS
# ------------------------------------------------------------

product_insights.to_csv(
    output_folder / "product_insights.csv",
    index=False
)


# ------------------------------------------------------------
# STEP 45 — SAVE PURCHASE INSIGHTS
# ------------------------------------------------------------

purchase_insights.to_csv(
    output_folder / "purchase_behavior_insights.csv",
    index=False
)


# ------------------------------------------------------------
# STEP 46 — SAVE CUSTOMER INSIGHTS
# ------------------------------------------------------------

customer_insights.to_csv(
    output_folder / "customer_value_insights.csv",
    index=False
)


# ------------------------------------------------------------
# STEP 47 — SAVE RFM INSIGHTS
# ------------------------------------------------------------

rfm_insights.to_csv(
    output_folder / "rfm_insights.csv",
    index=False
)


# ------------------------------------------------------------
# STEP 48 — SAVE CHURN INSIGHTS
# ------------------------------------------------------------

churn_insights.to_csv(
    output_folder / "churn_insights.csv",
    index=False
)


# ------------------------------------------------------------
# STEP 49 — SAVE PAYMENT INSIGHTS
# ------------------------------------------------------------

payment_insights.to_csv(
    output_folder / "payment_insights.csv",
    index=False
)


# ------------------------------------------------------------
# STEP 50 — SAVE RECOMMENDATIONS
# ------------------------------------------------------------

recommendations.to_csv(
    output_folder / "business_recommendations.csv",
    index=False
)


# ============================================================
# SECTION K — CREATE MASTER INSIGHTS TABLE
# ============================================================

# ------------------------------------------------------------
# STEP 51 — MASTER INSIGHTS
# ------------------------------------------------------------

master_insights = pd.DataFrame({
    "Category": [
        "Business",
        "Business",
        "Customer",
        "Customer",
        "Product",
        "Product",
        "Purchase Behavior",
        "Purchase Behavior",
        "Churn",
        "Churn",
        "RFM",
        "Payment"
    ],
    "Insight": [
        "Total Revenue",
        "Average Order Value",
        "Largest Customer Segment",
        "Top Customer",
        "Highest Revenue Category",
        "Most Purchased Category",
        "Best Purchase Day",
        "Peak Purchase Hour",
        "Churn Rate",
        "High-Value At-Risk Customers",
        "Highest Revenue Segment",
        "Most Used Payment Method"
    ],
    "Value": [
        total_revenue,
        average_order_value,
        largest_segment,
        top_customer_id,
        best_category,
        most_purchased_category,
        best_purchase_day,
        f"{int(best_purchase_hour)}:00",
        churn_rate,
        high_value_at_risk_count,
        highest_revenue_segment,
        most_used_payment
    ]
})

master_insights.to_csv(
    output_folder / "master_business_insights.csv",
    index=False
)


# ============================================================
# SECTION L — SAVE HIGH-VALUE AT-RISK CUSTOMERS
# ============================================================

if not high_value_at_risk.empty:

    high_value_at_risk.to_csv(
        output_folder / "high_value_at_risk_customers.csv",
        index=False
    )


# ============================================================
# SECTION M — FINAL OUTPUT CHECK
# ============================================================

# ------------------------------------------------------------
# STEP 52 — DISPLAY OUTPUT FILES
# ------------------------------------------------------------

print("\n====================================================")
print("STEP 8 COMPLETED SUCCESSFULLY")
print("====================================================\n")

print("Generated Files:\n")

for file in sorted(output_folder.iterdir()):

    print("✓", file.name)


# ------------------------------------------------------------
# STEP 53 — DISPLAY FINAL RECOMMENDATIONS
# ------------------------------------------------------------

print("\n====================================================")
print("FINAL 5 BUSINESS RECOMMENDATIONS")
print("====================================================\n")

display(recommendations)


# ------------------------------------------------------------
# STEP 54 — DISPLAY EXECUTIVE SUMMARY
# ------------------------------------------------------------

print(executive_summary)


# ============================================================
# END OF STEP 8
# ============================================================


================ BUSINESS KPIs ================



,Metric,Value
0,Total Customers,4.967300e+04
1,Total Transactions,2.500000e+05
2,Total Revenue,6.813427e+08
3,Average Order Value,2.725371e+03
4,Total Quantity,7.497240e+05
5,Average Quantity per Transaction,2.998896e+00
6,Total Returns,1.007690e+05
7,Return Rate (%),4.030760e+01
8,Churned Customers,9.942000e+03
9,Churn Rate (%),2.001490e+01



================ PRODUCT INSIGHTS ================



,Insight,Value,Metric
0,Highest Revenue Category,Books,2.049396e+08
1,Most Purchased Category,Clothing,7.505200e+04
2,Highest Return Rate Category,Home,4.041000e+01



================ PURCHASE BEHAVIOR INSIGHTS ================



...


================ PAYMENT INSIGHTS ================



,Insight,Value,Metric
0,Most Used Payment Method,Credit Card,100486
1,Highest Revenue Payment Method,Credit Card,274152396



5 ACTIONABLE BUSINESS RECOMMENDATIONS

1. [High] Launch a targeted retention campaign for high-value at-risk customers. The analysis identified 4,226 customers who have high monetary value but low recent engagement. Use personalized offers, reminders and exclusive benefits to encourage repeat purchases.

2. [High] Use RFM-based customer segmentation for personalized marketing. Prioritize the 'Champions' segment for loyalty programs and cross-selling, while using reactivation campaigns for low-recency segments.

3. [Medium] Focus inventory and marketing efforts on the 'Books' category because it generates the highest revenue in the analyzed dataset. Combine this category with cross-selling and bundle offers to increase customer order value.

4. [Medium] Schedule promotional campaigns around the strongest customer purchase period. The peak transaction period is Wednesday at approximately 17:00. Use this period for flash sales, personalized offers and campaign reminders.

5. [High] Imple

,Recommendation_ID,Priority,Recommendation
0,1,High,Launch a targeted retention campaign for high-...
1,2,High,Use RFM-based customer segmentation for person...
2,3,Medium,Focus inventory and marketing efforts on the '...
3,4,Medium,Schedule promotional campaigns around the stro...
4,5,High,Implement an early-warning churn system. The o...



CUSTOMER BEHAVIOR ANALYSIS — EXECUTIVE SUMMARY

BUSINESS PERFORMANCE
--------------------
Total Customers       : 49,673
Total Transactions    : 250,000
Total Revenue         : 681,342,683.00
Average Order Value   : 2,725.37
Total Quantity Sold   : 749,724
Return Rate           : 40.31%
Churn Rate            : 20.01%

CUSTOMER INSIGHTS
-----------------
Largest Customer Segment:
Hibernating

Highest Revenue Segment:
Champions

Top Customer:
36437

Average Customer Spend:
13,716.56

PRODUCT INSIGHTS
----------------
Highest Revenue Category:
Books

Most Purchased Category:
Clothing

Highest Return Rate Category:
Home

PURCHASE BEHAVIOR
-----------------
Best Purchase Day:
Wednesday

Peak Purchase Hour:
17:00

Best Time Period:
Night

Highest Revenue Month:
2020-12

PAYMENT BEHAVIOR
----------------
Most Used Payment Method:
Credit Card

Highest Revenue Payment Method:
Credit Card

CHURN INSIGHTS
--------------
Churned Customers:
9,942

Churn Rate:
20.01%

Revenue from Churned Customers